# Load packages and libraries

In [1]:
.libPaths()
assign(".lib.loc", "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library", envir = environment(.libPaths))
.libPaths()
# sessionInfo()


suppressMessages(library(dplyr)) 
suppressMessages(library(ggplot2)) 
suppressMessages(library(Matrix)) 
suppressMessages(library(data.table)) 
suppressMessages(library(ggpubr)) 
suppressMessages(library(ggplot2))
suppressMessages(library(pheatmap))
suppressMessages(library("cowplot"))
suppressMessages(library("RColorBrewer"))
suppressMessages(library("plyr"))
suppressMessages(library("forcats"))
suppressMessages(library('ggeasy'))
suppressMessages(library('dplyr'))
suppressMessages(library("svglite"))
suppressMessages(library("ape"))
suppressMessages(library("ggforce"))
suppressMessages(library("tidyr"))
suppressMessages(library("tibble")) 
library("ggrepel")

library("optparse")
suppressMessages(library("splitstackshape")) 
suppressMessages(library("ggupset"))



[1] "/group/soranzo/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

# Read DE results

In [2]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/DE_per_cluster/")

In [3]:
DE_results<-readRDS(file="DE_results_Diff_K562.rds")

cat("DE_results_0\n")
cat(str(DE_results))
cat("\n")



DE_results_0
'data.frame':	322320 obs. of  10 variables:
 $ gene          : chr  "PURPL" "GNB1" "RER1" "RPL22" ...
 $ baseMean      : num  9.2 5.98 6.08 11.84 32.94 ...
 $ log2FoldChange: num  8.62e-06 -2.07e-06 1.73e-06 -4.44e-06 -3.18e-06 ...
 $ lfcSE         : num  0.00144 0.00144 0.00144 0.00144 0.00144 ...
 $ pvalue        : num  9.88e-06 3.22e-01 4.99e-01 1.12e-01 2.55e-01 ...
 $ padj          : num  0.0159 0.9976 0.9976 0.9976 0.9976 ...
 $ contrast      : Ord.factor w/ 5 levels "Genotype_rs139141690_HET_vs_wt"<..: 1 1 1 1 1 1 1 1 1 1 ...
 $ identity      : Ord.factor w/ 13 levels "1"<"2"<"3"<"4"<..: 13 13 13 13 13 13 13 13 13 13 ...
 $ minuslog10padj: num  1.79762 0.00104 0.00104 0.00104 0.00104 ...
 $ abslogfc      : num  8.62e-06 2.07e-06 1.73e-06 4.44e-06 3.18e-06 ...



# Read normalised counts

In [4]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/DE_per_cluster/")
  
normalised_counts<-readRDS(file="norcounts_FINAL_Diff_K562.rds")

cat("normalised_counts_0\n")
cat(str(normalised_counts))
cat("\n")

normalised_counts_0
'data.frame':	3175714 obs. of  5 variables:
 $ gene      : chr  "GNB1" "RER1" "RPL22" "RERE" ...
 $ count     : num  10.75 6.95 10.88 47.57 13.5 ...
 $ identity  : Ord.factor w/ 13 levels "1"<"2"<"3"<"4"<..: 13 13 13 13 13 13 13 13 13 13 ...
 $ time_point: Ord.factor w/ 4 levels "24_hours"<"48_hours"<..: 1 1 1 1 1 1 1 1 1 1 ...
 $ clone_line: Ord.factor w/ 11 levels "wt_1"<"wt_2"<..: 8 8 8 8 8 8 8 8 8 8 ...



# Read ORA annotations

In [5]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/DE_per_cluster/")

In [6]:
annotation_file<-read.table(file="genes_ORA_annotated_Diff_K562.tsv", sep="\t", header=T)

cat("annotation_file_0\n")
cat(str(annotation_file))
cat("\n")


annotation_file_gene_names<-annotation_file[-grep("^[0-9]+$",annotation_file$gene),]

annotation_file_long<-unique(as.data.frame(cSplit(annotation_file_gene_names,sep = '|', direction = "long",
                                                    splitCols = "other"),stringsAsFactors=F))
  
cat("annotation_file_long_0\n")
cat(str(annotation_file_long))
cat("\n")

annotation_file_0
'data.frame':	1959 obs. of  5 variables:
 $ gene         : chr  "10006" "10006" "100133941" "10049" ...
 $ identity     : int  3 1 1 1 3 3 1 3 1 3 ...
 $ diffexpressed: chr  "UP" "UP" "DOWN" "DOWN" ...
 $ other        : chr  "TRAVAGLINI_LUNG_PLATELET_MEGAKARYOCYTE_CELL" "TRAVAGLINI_LUNG_PLATELET_MEGAKARYOCYTE_CELL" "DUNNE_TARGETS_OF_AML1_MTG8_FUSION_UP|GSE10325_BCELL_VS_MYELOID_UP" "TRAVAGLINI_LUNG_PLATELET_MEGAKARYOCYTE_CELL" ...
 $ TF_targets   : chr  NA NA NA NA ...



Warning message in type.convert.default(unlist(x, use.names = FALSE)):
“'as.is' should be specified by the caller; using TRUE”


annotation_file_long_0
'data.frame':	2850 obs. of  5 variables:
 $ gene         : chr  "A2M" "A2M" "ABCA8" "ABCB1" ...
 $ identity     : int  3 3 3 7 7 7 7 3 3 3 ...
 $ diffexpressed: chr  "UP" "UP" "UP" "DOWN" ...
 $ other        : chr  "GOCC_PLATELET_ALPHA_GRANULE" "GOCC_PLATELET_ALPHA_GRANULE_LUMEN" NA "Ageing" ...
 $ TF_targets   : chr  NA NA "Dorothea_ABC_GATA2_targets" NA ...



In [7]:
length(grep("^[0-9]+$",annotation_file$gene))

[1] 741

# Selected annotations

In [83]:
annotations<-c("GOBP_MEGAKARYOCYTE_DIFFERENTIATION","HP_ABNORMAL_PLATELET_VOLUME","HP_INCREASED_MEAN_PLATELET_VOLUME","WP_PI3KAKT_SIGNALING","AKT_UP_MTOR_DN.V1_UP")

In [84]:
annotation_file_long_sel<-annotation_file_long[grep(paste(annotations, collapse="|"), annotation_file_long$other),]

cat("annotation_file_long_sel_0\n")
cat(str(annotation_file_long_sel))
cat("\n")
cat(str(unique(annotation_file_long_sel$gene)))
cat("\n")

cat(sprintf(as.character(unique(annotation_file_long_sel$gene))))
cat("\n")
cat("\n")

cat(sprintf(as.character(unique(annotation_file_long_sel$other))))
cat("\n")


annotation_file_long_sel_0
'data.frame':	71 obs. of  5 variables:
 $ gene         : chr  "ABI1" "ABI1" "ACTB" "ACTB" ...
 $ identity     : int  3 1 1 1 3 1 3 1 7 4 ...
 $ diffexpressed: chr  "UP" "UP" "DOWN" "DOWN" ...
 $ other        : chr  "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "HP_ABNORMAL_PLATELET_VOLUME" "HP_INCREASED_MEAN_PLATELET_VOLUME" ...
 $ TF_targets   : chr  "PU1_Q6|RGAGGAARY_PU1_Q6" "PU1_Q6|RGAGGAARY_PU1_Q6" "Dorothea_ABCDE_CUX1_targets|Dorothea_ABCD_CUX1_targets" "Dorothea_ABCDE_CUX1_targets|Dorothea_ABCD_CUX1_targets" ...

 chr [1:43] "ABI1" "ACTB" "ADAM10" "ANGPT1" "ARRDC4" "ATF3" "C3orf52" ...

ABI1 ACTB ADAM10 ANGPT1 ARRDC4 ATF3 C3orf52 CD55 CIR1 DHCR24 DIAPH1 FGF13 FLNA FOXO3 FYB1 GFI1B GUSB HOXB3 IL1R1 ITGA2B KIT LAMA2 MAPK1 MEF2C MEIS1 MYB PIK3CA PIP4K2A PKN2 PLAGL1 PPP2R3A RBL2 SGPL1 SH2B3 SLC44A1 SLC6A6 SOS1 SUCO TOR3A TPM4 TUBB1 WASF2 ZFPM1

GOBP_MEGAKARYOCYTE_DIFFERENTIATION HP_ABNORMAL_PLATELET_VOLUME HP_INCREASED_MEAN_PLATE

In [85]:
annotation_file_long_sel_for_collapse<-unique(annotation_file_long_sel[,which(colnames(annotation_file_long_sel)%in%c('gene','other'))])

annotation_file_long_sel_for_collapse<-annotation_file_long_sel_for_collapse[order(annotation_file_long_sel_for_collapse$gene, annotation_file_long_sel_for_collapse$other),]

cat("annotation_file_long_sel_for_collapse_0\n")
cat(str(annotation_file_long_sel_for_collapse))
cat("\n")

annotation_file_long_sel_for_collapse_0
'data.frame':	50 obs. of  2 variables:
 $ gene : chr  "ABI1" "ACTB" "ACTB" "ADAM10" ...
 $ other: chr  "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "HP_ABNORMAL_PLATELET_VOLUME" "HP_INCREASED_MEAN_PLATELET_VOLUME" "AKT_UP_MTOR_DN.V1_UP" ...



In [86]:
annotation_file_long_sel_for_collapse.dt<-data.table(annotation_file_long_sel_for_collapse, key='gene')
      
annotation_file_sel_collapsed<-unique(as.data.frame(annotation_file_long_sel_for_collapse.dt[,.(other_string=paste(other, collapse="|")), by=key(annotation_file_long_sel_for_collapse.dt)], stringsAsFactors=F))



cat("annotation_file_sel_collapsed_0\n")
cat(str(annotation_file_sel_collapsed))
cat("\n")
cat(names(summary(as.factor(annotation_file_sel_collapsed$other_string))))
cat("\n")
cat(summary(as.factor(annotation_file_sel_collapsed$other_string)))
cat("\n")
      

annotation_file_sel_collapsed_0
'data.frame':	43 obs. of  2 variables:
 $ gene        : chr  "ABI1" "ACTB" "ADAM10" "ANGPT1" ...
 $ other_string: chr  "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "HP_ABNORMAL_PLATELET_VOLUME|HP_INCREASED_MEAN_PLATELET_VOLUME" "AKT_UP_MTOR_DN.V1_UP" "WP_PI3KAKT_SIGNALING" ...

AKT_UP_MTOR_DN.V1_UP GOBP_MEGAKARYOCYTE_DIFFERENTIATION HP_ABNORMAL_PLATELET_VOLUME HP_ABNORMAL_PLATELET_VOLUME|HP_INCREASED_MEAN_PLATELET_VOLUME WP_PI3KAKT_SIGNALING
16 9 1 7 10


# Selected genes

In [87]:
genes<-c("CUX1","RUNX1","EZH2","XRCC2")

annotation_of_selected_genes<-c("CUX1","RUNX1",rep("EZH2 and targets",2))


selected_genes_df<-as.data.frame(cbind(genes,annotation_of_selected_genes))

colnames(selected_genes_df)<-c("gene","other_string")

cat("selected_genes_df_0\n")
cat(str(selected_genes_df))
cat("\n")

selected_genes_df_0
'data.frame':	4 obs. of  2 variables:
 $ gene        : chr  "CUX1" "RUNX1" "EZH2" "XRCC2"
 $ other_string: chr  "CUX1" "RUNX1" "EZH2 and targets" "EZH2 and targets"



# Put together and select from DE genes

In [88]:
REP_genes<-rbind(selected_genes_df,annotation_file_sel_collapsed)

cat("REP_genes_0\n")
cat(str(REP_genes))
cat("\n")
cat(str(unique(REP_genes$gene)))
cat("\n")

REP_genes_0
'data.frame':	47 obs. of  2 variables:
 $ gene        : chr  "CUX1" "RUNX1" "EZH2" "XRCC2" ...
 $ other_string: chr  "CUX1" "RUNX1" "EZH2 and targets" "EZH2 and targets" ...

 chr [1:47] "CUX1" "RUNX1" "EZH2" "XRCC2" "ABI1" "ACTB" "ADAM10" "ANGPT1" ...



In [89]:
str(REP_genes)

'data.frame':	47 obs. of  2 variables:
 $ gene        : chr  "CUX1" "RUNX1" "EZH2" "XRCC2" ...
 $ other_string: chr  "CUX1" "RUNX1" "EZH2 and targets" "EZH2 and targets" ...


In [91]:
names(summary(as.factor(REP_genes$other_string)))

[1] "AKT_UP_MTOR_DN.V1_UP"                                         
[2] "CUX1"                                                         
[3] "EZH2 and targets"                                             
[4] "GOBP_MEGAKARYOCYTE_DIFFERENTIATION"                           
[5] "HP_ABNORMAL_PLATELET_VOLUME"                                  
[6] "HP_ABNORMAL_PLATELET_VOLUME|HP_INCREASED_MEAN_PLATELET_VOLUME"
[7] "RUNX1"                                                        
[8] "WP_PI3KAKT_SIGNALING"

In [92]:
REP_genes$Gene_class<-NA

In [94]:
REP_genes$Gene_class<-factor(REP_genes$other_string,
                            levels=c('CUX1',
                                    'RUNX1',
                                    'EZH2 and targets',
                                    'GOBP_MEGAKARYOCYTE_DIFFERENTIATION',
                                    'HP_ABNORMAL_PLATELET_VOLUME',
                                    'HP_ABNORMAL_PLATELET_VOLUME|HP_INCREASED_MEAN_PLATELET_VOLUME',
                                    'WP_PI3KAKT_SIGNALING',
                                    'AKT_UP_MTOR_DN.V1_UP'),
                            ordered=T)

In [95]:
str(REP_genes)

'data.frame':	47 obs. of  3 variables:
 $ gene        : chr  "CUX1" "RUNX1" "EZH2" "XRCC2" ...
 $ other_string: chr  "CUX1" "RUNX1" "EZH2 and targets" "EZH2 and targets" ...
 $ Gene_class  : Ord.factor w/ 8 levels "CUX1"<"RUNX1"<..: 1 2 3 3 4 6 8 7 8 8 ...


In [96]:
DE_results_sel<-DE_results[which(DE_results$gene%in%REP_genes$gene &
                                DE_results$identity%in%c('3','1')),]

cat("DE_results_sel_0\n")
cat(str(DE_results_sel))
cat("\n")
cat(str(unique(DE_results_sel$gene)))
cat("\n")


DE_results_sel_0
'data.frame':	368 obs. of  10 variables:
 $ gene          : chr  "PIP4K2A" "C3orf52" "MEF2C" "WASF2" ...
 $ baseMean      : num  301.1 6.6 686.9 137.4 33.6 ...
 $ log2FoldChange: num  0.21776 0.00297 0.0066 0.00122 -0.00421 ...
 $ lfcSE         : num  0.1375 0.0209 0.022 0.0203 0.0211 ...
 $ pvalue        : num  0.00255 0.01433 0.02977 0.75244 0.09663 ...
 $ padj          : num  0.758 0.867 0.986 0.996 0.996 ...
 $ contrast      : Ord.factor w/ 5 levels "Genotype_rs139141690_HET_vs_wt"<..: 1 1 1 1 1 1 1 1 1 1 ...
 $ identity      : Ord.factor w/ 13 levels "1"<"2"<"3"<"4"<..: 3 3 3 3 3 3 3 3 3 3 ...
 $ minuslog10padj: num  0.1203 0.0618 0.0063 0.0016 0.0016 ...
 $ abslogfc      : num  0.21776 0.00297 0.0066 0.00122 0.00421 ...

 chr [1:47] "PIP4K2A" "C3orf52" "MEF2C" "WASF2" "DHCR24" "PKN2" "SUCO" ...



# Heatmap of replicas

In [97]:
DEBUG <- 1

In [98]:
 REP<-normalised_counts[which(normalised_counts$gene%in%DE_results_sel$gene &
                                     normalised_counts$identity%in%DE_results_sel$identity),]
      
      if(DEBUG == 1){
        
        cat("REP_0\n")
        cat(str(REP))
        cat("\n")
      }

REP_0
'data.frame':	3860 obs. of  5 variables:
 $ gene      : chr  "WASF2" "DHCR24" "PKN2" "SUCO" ...
 $ count     : num  156.5 26.1 187.9 107 38.7 ...
 $ identity  : Ord.factor w/ 13 levels "1"<"2"<"3"<"4"<..: 3 3 3 3 3 3 3 3 3 3 ...
 $ time_point: Ord.factor w/ 4 levels "24_hours"<"48_hours"<..: 1 1 1 1 1 1 1 1 1 1 ...
 $ clone_line: Ord.factor w/ 11 levels "wt_1"<"wt_2"<..: 8 8 8 8 8 8 8 8 8 8 ...



In [99]:
 REP_wide<-as.data.frame(pivot_wider(REP, id_cols=c('gene'),
                                           names_from=c('identity',"clone_line","time_point"),
                                           values_from='count',
                                           names_sep='|'), stringsAsFactors=F)

   if(DEBUG == 1){

     cat("REP_wide_0\n")
     #cat(str(REP_wide))
     cat("\n")
   }

   GeneEXP_matrix<-as.matrix(REP_wide[,-which(colnames(REP_wide)%in%c('gene'))])

   row.names(GeneEXP_matrix)<-REP_wide$gene

   if(DEBUG == 1){

     cat("GeneEXP_matrix_0\n")
     cat(str(GeneEXP_matrix))
     cat("\n")
   }


REP_wide_0

GeneEXP_matrix_0
 num [1:47, 1:84] 156.5 26.1 187.9 107 38.7 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:47] "WASF2" "DHCR24" "PKN2" "SUCO" ...
  ..$ : chr [1:84] "3|Del_16bp_1|24_hours" "3|Del_80bp_1|24_hours" "3|Del_80bp_2|24_hours" "3|Del_80bp_3|24_hours" ...



## Annotation col

In [100]:
annotation_col<- data.frame(matrix(vector(), length(colnames(GeneEXP_matrix)), 3,
                                          dimnames=list(c(),
                                                        c("identity","time_point","clone_line"))),stringsAsFactors=F)

       row.names(annotation_col)<-colnames(GeneEXP_matrix)

 if(DEBUG == 1){
         cat("annotation_col_0\n")
         cat(str(annotation_col))
         cat("\n")
         cat(str(row.names(annotation_col)))
         cat("\n")

       }


       annotation_col$identity<-gsub("\\|.+$","", row.names(annotation_col))
       annotation_col$clone_line<-gsub("^[^\\|]+\\|","", row.names(annotation_col))
       annotation_col$clone_line<-gsub("\\|.+$","", annotation_col$clone_line)

       annotation_col$time_point<-gsub("^[^\\|]+\\|[^\\|]+\\|","", row.names(annotation_col))

       if(DEBUG == 1){
         cat("annotation_col_1\n")
         cat(str(annotation_col))
         cat("\n")
       }

       annotation_col$Genotype<-NA

       annotation_col$Genotype[which(annotation_col$clone_line%in%c('wt_1','wt_2','wt_3'))]<-'wt'
       annotation_col$Genotype[which(annotation_col$clone_line%in%c('rs139141690_HET_1'))]<-'rs139141690_HET'
       annotation_col$Genotype[which(annotation_col$clone_line%in%c('rs139141690_1','rs139141690_2','rs139141690_3'))]<-'rs139141690'
       annotation_col$Genotype[which(annotation_col$clone_line%in%c('Del_16bp_1'))]<-'Del_16bp'
       annotation_col$Genotype[which(annotation_col$clone_line%in%c('Del_80bp_1','Del_80bp_2','Del_80bp_3'))]<-'Del_80bp'

       if(DEBUG == 1){
         cat("annotation_col_PRE\n")
         cat(str(annotation_col))
         cat("\n")
         names(summary(as.factor(annotation_col$Genotype)))
         cat("\n")
         names(summary(as.factor(annotation_col$clone_line)))
         cat("\n")
         names(summary(as.factor(annotation_col$identity)))
         cat("\n")
       }

  annotation_col$Genotype<-factor(annotation_col$Genotype,
                                       levels=c('wt','rs139141690_HET','rs139141690','Del_16bp','Del_80bp'),
                                       ordered=T)

       annotation_col$clone_line<-factor(annotation_col$clone_line,
                                         levels=levels(normalised_counts$clone_line),
                                         ordered=T)

       annotation_col$identity<-factor(annotation_col$identity,
                                       levels=levels(DE_results_sel$identity),
                                       ordered=T)

       annotation_col$time_point<-factor(annotation_col$time_point,
                                         levels=levels(normalised_counts$time_point),
                                         ordered=T)

       if(DEBUG == 1){
         cat("annotation_col_POST\n")
         cat(str(annotation_col))
         cat("\n")
         names(summary(as.factor(annotation_col$Genotype)))
         cat("\n")
         names(summary(as.factor(annotation_col$clone_line)))
         cat("\n")
         names(summary(as.factor(annotation_col$identity)))
         cat("\n")
       }



annotation_col_0
'data.frame':	84 obs. of  3 variables:
 $ identity  : logi  NA NA NA NA NA NA ...
 $ time_point: logi  NA NA NA NA NA NA ...
 $ clone_line: logi  NA NA NA NA NA NA ...

 chr [1:84] "3|Del_16bp_1|24_hours" "3|Del_80bp_1|24_hours" ...

annotation_col_1
'data.frame':	84 obs. of  3 variables:
 $ identity  : chr  "3" "3" "3" "3" ...
 $ time_point: chr  "24_hours" "24_hours" "24_hours" "24_hours" ...
 $ clone_line: chr  "Del_16bp_1" "Del_80bp_1" "Del_80bp_2" "Del_80bp_3" ...

annotation_col_PRE
'data.frame':	84 obs. of  4 variables:
 $ identity  : chr  "3" "3" "3" "3" ...
 $ time_point: chr  "24_hours" "24_hours" "24_hours" "24_hours" ...
 $ clone_line: chr  "Del_16bp_1" "Del_80bp_1" "Del_80bp_2" "Del_80bp_3" ...
 $ Genotype  : chr  "Del_16bp" "Del_80bp" "Del_80bp" "Del_80bp" ...




annotation_col_POST
'data.frame':	84 obs. of  4 variables:
 $ identity  : Ord.factor w/ 13 levels "1"<"2"<"3"<"4"<..: 3 3 3 3 3 3 3 3 3 3 ...
 $ time_point: Ord.factor w/ 4 levels "24_hours"<"48

## Annotation row

In [101]:
cat("REP_genes_remember\n")
cat(str(REP_genes))
cat("\n")

REP_genes_remember
'data.frame':	47 obs. of  3 variables:
 $ gene        : chr  "CUX1" "RUNX1" "EZH2" "XRCC2" ...
 $ other_string: chr  "CUX1" "RUNX1" "EZH2 and targets" "EZH2 and targets" ...
 $ Gene_class  : Ord.factor w/ 8 levels "CUX1"<"RUNX1"<..: 1 2 3 3 4 6 8 7 8 8 ...



In [102]:
annotation_row = data.frame(GeneClass = REP_genes$Gene_class)
      
rownames(annotation_row) = REP_genes$gene

if(DEBUG == 1){
cat("annotation_row_0\n")
cat(str(annotation_row))
cat("\n")
cat(sprintf(as.character(names(summary(as.factor(annotation_row$GeneClass))))))
cat("\n")
cat(sprintf(as.character(summary(as.factor(annotation_row$GeneClass)))))
cat("\n")
}

annotation_row_0
'data.frame':	47 obs. of  1 variable:
 $ GeneClass: Ord.factor w/ 8 levels "CUX1"<"RUNX1"<..: 1 2 3 3 4 6 8 7 8 8 ...

CUX1 RUNX1 EZH2 and targets GOBP_MEGAKARYOCYTE_DIFFERENTIATION HP_ABNORMAL_PLATELET_VOLUME HP_ABNORMAL_PLATELET_VOLUME|HP_INCREASED_MEAN_PLATELET_VOLUME WP_PI3KAKT_SIGNALING AKT_UP_MTOR_DN.V1_UP
1 1 2 9 1 7 10 16


## vector colors

In [103]:
vector_colors_clone_line<- c(brewer.pal(9, "Greens")[c(5,6,7)],
                                                  brewer.pal(9, "YlOrRd")[c(2)],
                                                  brewer.pal(9, "Reds")[c(5,6,7)],
                                                  brewer.pal(9, "Purples")[c(7)],
                                                  brewer.pal(9, "Blues")[c(4,5,6)])

        names(vector_colors_clone_line)<-levels(annotation_col$clone_line)

        vector_colors_Genotype<-c(brewer.pal(9, "Greens")[c(5)],
                                  brewer.pal(9, "YlOrRd")[c(2)],
                                    brewer.pal(9, "Reds")[c(5)],
                                    brewer.pal(9, "Purples")[c(7)],
                                    brewer.pal(9, "Blues")[c(4)])

        names(vector_colors_Genotype)<-levels(annotation_col$Genotype)

        vector_colors_time_point<-c(brewer.pal(9, "Greys")[c(1)],
                                    brewer.pal(9, "Greys")[c(3)],
                                    brewer.pal(9, "Greys")[c(5)],
                                    brewer.pal(9, "Greys")[c(7)])

        names(vector_colors_time_point)<-levels(annotation_col$time_point)




        vector_colors_seurat_clusters<-c(brewer.pal(9, "YlOrRd")[c(7)],
                                         brewer.pal(9, "Blues")[c(5)],
                                         brewer.pal(9, "RdPu")[c(6)],
                                         brewer.pal(9, "Blues")[c(4)],
                                         brewer.pal(9, "Greens")[c(6)],
                                         brewer.pal(9, "YlOrRd")[c(6)],
                                         brewer.pal(9, "Blues")[c(3)],
                                         brewer.pal(9, "RdPu")[c(5)],
                                         brewer.pal(9, "Greens")[c(5)],
                                         brewer.pal(9, "Greens")[c(4)],
                                         brewer.pal(9, "Blues")[c(2)],
                                         brewer.pal(9, "RdPu")[c(4)],
                                         brewer.pal(9, "RdPu")[c(3)])

        values<-vector_colors_seurat_clusters


        # harcoded!!!!

        names<-levels(DE_results_sel$identity)

values<-values[1:length(names)]


        vector_colors_identity<-setNames(values, names)

        if(DEBUG == 1){
          cat("vector_colors_identity_0\n")
          cat(str(vector_colors_identity))
          cat("\n")
        }

        vector_colors_identity<-vector_colors_identity[which(names(vector_colors_identity)%in%levels(annotation_col$identity))]



        if(DEBUG == 1){
          cat("vector_colors_identity_1\n")
          cat(str(vector_colors_identity))
          cat("\n")
        }



vector_colors_identity_0
 Named chr [1:13] "#E31A1C" "#6BAED6" "#DD3497" "#9ECAE1" "#41AB5D" ...
 - attr(*, "names")= chr [1:13] "1" "2" "3" "4" ...

vector_colors_identity_1
 Named chr [1:13] "#E31A1C" "#6BAED6" "#DD3497" "#9ECAE1" "#41AB5D" ...
 - attr(*, "names")= chr [1:13] "1" "2" "3" "4" ...



### Gene class vector

In [104]:
cat("REP_genes_remember\n")
cat(str(REP_genes))
cat("\n")

REP_genes_remember
'data.frame':	47 obs. of  3 variables:
 $ gene        : chr  "CUX1" "RUNX1" "EZH2" "XRCC2" ...
 $ other_string: chr  "CUX1" "RUNX1" "EZH2 and targets" "EZH2 and targets" ...
 $ Gene_class  : Ord.factor w/ 8 levels "CUX1"<"RUNX1"<..: 1 2 3 3 4 6 8 7 8 8 ...



In [105]:
n_levels<-length(levels(REP_genes$Gene_class))

GeneClass_vector<-rep("0",n_levels)
      
      for(iteration_n_levels in 1:length(GeneClass_vector)){
        
        if(iteration_n_levels <= 8){
          
          GeneClass_vector[iteration_n_levels]<-brewer.pal(8, "Dark2")[iteration_n_levels]
          
        }else{
          
          GeneClass_vector[iteration_n_levels]<-brewer.pal(12, "Set3")[(iteration_n_levels - (8))]
          
        }#iteration_n_levels <= 8
        
        
        
      }#iteration_n_levels in 1:length(GeneClass_vector)
      
      names(GeneClass_vector)<-levels(REP_genes$Gene_class)
      
      if(DEBUG == 1){
        cat("GeneClass_vector_0\n")
        cat(str(GeneClass_vector))
        cat("\n")
      }

GeneClass_vector_0
 Named chr [1:8] "#1B9E77" "#D95F02" "#7570B3" "#E7298A" "#66A61E" ...
 - attr(*, "names")= chr [1:8] "CUX1" "RUNX1" "EZH2 and targets" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" ...



## Ann colors

In [106]:
ann_colors <- list( clone_line = vector_colors_clone_line,
                           Genotype = vector_colors_Genotype,
                           time_point = vector_colors_time_point,
                           identity =vector_colors_identity,
                          GeneClass = GeneClass_vector)


       if(DEBUG == 1){
         cat("ann_colors_0\n")
         cat(str(ann_colors))
         cat("\n")
       }


ann_colors_0
List of 5
 $ clone_line: Named chr [1:11] "#74C476" "#41AB5D" "#238B45" "#FFEDA0" ...
  ..- attr(*, "names")= chr [1:11] "wt_1" "wt_2" "wt_3" "rs139141690_HET_1" ...
 $ Genotype  : Named chr [1:5] "#74C476" "#FFEDA0" "#FB6A4A" "#6A51A3" ...
  ..- attr(*, "names")= chr [1:5] "wt" "rs139141690_HET" "rs139141690" "Del_16bp" ...
 $ time_point: Named chr [1:4] "#FFFFFF" "#D9D9D9" "#969696" "#525252"
  ..- attr(*, "names")= chr [1:4] "24_hours" "48_hours" "72_hours" "96_hours"
 $ identity  : Named chr [1:13] "#E31A1C" "#6BAED6" "#DD3497" "#9ECAE1" ...
  ..- attr(*, "names")= chr [1:13] "1" "2" "3" "4" ...
 $ GeneClass : Named chr [1:8] "#1B9E77" "#D95F02" "#7570B3" "#E7298A" ...
  ..- attr(*, "names")= chr [1:8] "CUX1" "RUNX1" "EZH2 and targets" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" ...



## graph

In [107]:
 path_graphs = paste("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/DE_per_cluster/",'test_graphs','/',sep='')
  
  if (file.exists(path_graphs)){
    
    
  }else{
    
    dir.create(file.path(path_graphs))
    
  }#path_Downstream_analysis

NULL

In [108]:
 heatmap<-pheatmap(GeneEXP_matrix, display_numbers = FALSE,
                             show_colnames=FALSE,
                             angle_col = "0",
                             clustering_method="ward.D2",
                             fontsize_row = 8,
                             fontsize_col = 8,
                             breaks=seq(-2,2,length.out=101),
                             color=colorRampPalette(c("blue","white","red"))(100),
                             scale="row",
                             cluster_cols=TRUE,
                             border_color='black',
                             treeheight_row=70, treeheight_col=70, cutree_cols=7,
                             annotation_col = annotation_col,
                             annotation_row = annotation_row,
                             annotation_colors = ann_colors)

setwd(path_graphs)

svgname<-paste("Heatmap_selected",".svg",sep='')

ggsave(svgname,plot=heatmap, device ='svg', width=13, height=13)


ERROR: Error in value[[3L]](cond): no active device and default getOption("device") is invalid


plot without title

# Logpval plot

## Order from heatmap

In [109]:
selected_genes_after_heatmap_clustering<-heatmap$tree_row$labels[heatmap$tree_row$order]
        
        
cat("selected_genes_after_heatmap_clustering_0\n")
cat(str(selected_genes_after_heatmap_clustering))
cat("\n")



logpval_df<-DE_results_sel[which(DE_results_sel$gene%in%selected_genes_after_heatmap_clustering),]


if(DEBUG == 1){
  cat("logpval_df_0\n")
  cat(str(logpval_df))
  cat("\n")
}

logpval_df$gene<-factor(logpval_df$gene, levels=rev(selected_genes_after_heatmap_clustering), ordered=T)



logpval_df$SIG<-NA

logpval_df$SIG[which(logpval_df$minuslog10padj >= 1.3)]<-'YES'
logpval_df$SIG[which(logpval_df$minuslog10padj < 1.3)]<-'NO'


logpval_df$SIG<-factor(logpval_df$SIG, levels=c('NO','YES'), ordered=T)

if(DEBUG == 1){
  cat("logpval_df_1\n")
  cat(str(logpval_df))
  cat("\n")
}

selected_genes_after_heatmap_clustering_0
 chr [1:47] "HOXB3" "CIR1" "ARRDC4" "FOXO3" "SUCO" "SOS1" "ADAM10" "FYB1" ...

logpval_df_0
'data.frame':	368 obs. of  10 variables:
 $ gene          : chr  "PIP4K2A" "C3orf52" "MEF2C" "WASF2" ...
 $ baseMean      : num  301.1 6.6 686.9 137.4 33.6 ...
 $ log2FoldChange: num  0.21776 0.00297 0.0066 0.00122 -0.00421 ...
 $ lfcSE         : num  0.1375 0.0209 0.022 0.0203 0.0211 ...
 $ pvalue        : num  0.00255 0.01433 0.02977 0.75244 0.09663 ...
 $ padj          : num  0.758 0.867 0.986 0.996 0.996 ...
 $ contrast      : Ord.factor w/ 5 levels "Genotype_rs139141690_HET_vs_wt"<..: 1 1 1 1 1 1 1 1 1 1 ...
 $ identity      : Ord.factor w/ 13 levels "1"<"2"<"3"<"4"<..: 3 3 3 3 3 3 3 3 3 3 ...
 $ minuslog10padj: num  0.1203 0.0618 0.0063 0.0016 0.0016 ...
 $ abslogfc      : num  0.21776 0.00297 0.0066 0.00122 0.00421 ...

logpval_df_1
'data.frame':	368 obs. of  11 variables:
 $ gene          : Ord.factor w/ 47 levels "TOR3A"<"DHCR24"<..: 29 15 34 12

In [110]:
logpval_dotplot<-ggplot(data=logpval_df,
                         aes(y=gene,
                             x=contrast))+
   geom_point(aes(size=minuslog10padj,
                  color=SIG,
                  fill=log2FoldChange),
              stroke=1, shape=21)+
   scale_size(range = c(0,6), name='-log10pval')+
   scale_y_discrete(name=NULL)+
   scale_x_discrete(name=NULL)+
   scale_fill_gradient2(
     low = "blue",
     mid = "white",
     high = "red",
     midpoint = 0)+
   scale_color_manual(name='p < 0.05',values=c('gray','black'))

logpval_dotplot<-logpval_dotplot+
                     theme_cowplot(font_size = 2,
                  font_family = "sans")+
                    facet_grid(. ~ identity, scales='free_x', space='free_x', switch="y")+
                    theme( strip.background = element_blank(),
                           strip.placement = "outside",
                           strip.text = element_text(size=5,color="black", family="sans"),
                           panel.spacing = unit(0.2, "lines"),
                           panel.background=element_rect(fill="white"),
                           panel.border=element_rect(colour="white",size=0,5),
                           panel.grid.major = element_blank(),
                           panel.grid.minor = element_blank())+
            theme_classic()+
           theme(axis.title.y=element_blank(),
                 axis.title.x=element_blank(),
                 axis.text.y=element_text(size=8, color="black", family="sans"),
                 axis.text.x=element_text(angle=45,size=8,vjust=1, hjust=1, color="black", family="sans"),
                 axis.line.x = element_line(size = 0.4),
                 axis.ticks.x = element_blank(),
                 axis.ticks.y = element_line(size = 0.4),
                 axis.line.y = element_line(size = 0.4))+
           theme(legend.title = element_text(size=12),
                 legend.text = element_text(size=8),
                 legend.key.size = unit(0.5, 'cm'), #change legend key size
                 legend.key.height = unit(0.5, 'cm'), #change legend key height
                 legend.key.width = unit(0.5, 'cm'), #change legend key width
                 legend.position="right")+
           ggeasy::easy_center_title()

setwd(path_graphs)

svgname<-paste("Logpval_selected",".svg",sep='')

ggsave(svgname,plot=logpval_dotplot, device ='svg', width=13, height=13)

## Order from REP genes

In [112]:
order_from_REP<-REP_genes$gene
        
        
cat("order_from_REP_0\n")
cat(str(order_from_REP))
cat("\n")



logpval_df<-DE_results_sel[which(DE_results_sel$gene%in%order_from_REP),]


if(DEBUG == 1){
  cat("logpval_df_0\n")
  cat(str(logpval_df))
  cat("\n")
}

logpval_df$gene<-factor(logpval_df$gene, levels=rev(order_from_REP), ordered=T)



logpval_df$SIG<-NA

logpval_df$SIG[which(logpval_df$minuslog10padj >= 1.3)]<-'YES'
logpval_df$SIG[which(logpval_df$minuslog10padj < 1.3)]<-'NO'


logpval_df$SIG<-factor(logpval_df$SIG, levels=c('NO','YES'), ordered=T)

if(DEBUG == 1){
  cat("logpval_df_1\n")
  cat(str(logpval_df))
  cat("\n")
}

order_from_REP_0
 chr [1:47] "CUX1" "RUNX1" "EZH2" "XRCC2" "ABI1" "ACTB" "ADAM10" "ANGPT1" ...

logpval_df_0
'data.frame':	368 obs. of  10 variables:
 $ gene          : chr  "PIP4K2A" "C3orf52" "MEF2C" "WASF2" ...
 $ baseMean      : num  301.1 6.6 686.9 137.4 33.6 ...
 $ log2FoldChange: num  0.21776 0.00297 0.0066 0.00122 -0.00421 ...
 $ lfcSE         : num  0.1375 0.0209 0.022 0.0203 0.0211 ...
 $ pvalue        : num  0.00255 0.01433 0.02977 0.75244 0.09663 ...
 $ padj          : num  0.758 0.867 0.986 0.996 0.996 ...
 $ contrast      : Ord.factor w/ 5 levels "Genotype_rs139141690_HET_vs_wt"<..: 1 1 1 1 1 1 1 1 1 1 ...
 $ identity      : Ord.factor w/ 13 levels "1"<"2"<"3"<"4"<..: 3 3 3 3 3 3 3 3 3 3 ...
 $ minuslog10padj: num  0.1203 0.0618 0.0063 0.0016 0.0016 ...
 $ abslogfc      : num  0.21776 0.00297 0.0066 0.00122 0.00421 ...

logpval_df_1
'data.frame':	368 obs. of  11 variables:
 $ gene          : Ord.factor w/ 47 levels "ZFPM1"<"WASF2"<..: 16 37 20 2 34 15 6 5 36 38 ...
 $ bas

In [113]:
logpval_dotplot<-ggplot(data=logpval_df,
                         aes(y=gene,
                             x=contrast))+
   geom_point(aes(size=minuslog10padj,
                  color=SIG,
                  fill=log2FoldChange),
              stroke=1, shape=21)+
   scale_size(range = c(0,6), name='-log10pval')+
   scale_y_discrete(name=NULL)+
   scale_x_discrete(name=NULL)+
   scale_fill_gradient2(
     low = "blue",
     mid = "white",
     high = "red",
     midpoint = 0)+
   scale_color_manual(name='p < 0.05',values=c('gray','black'))

logpval_dotplot<-logpval_dotplot+
                     theme_cowplot(font_size = 2,
                  font_family = "sans")+
                    facet_grid(. ~ identity, scales='free_x', space='free_x', switch="y")+
                    theme( strip.background = element_blank(),
                           strip.placement = "outside",
                           strip.text = element_text(size=5,color="black", family="sans"),
                           panel.spacing = unit(0.2, "lines"),
                           panel.background=element_rect(fill="white"),
                           panel.border=element_rect(colour="white",size=0,5),
                           panel.grid.major = element_blank(),
                           panel.grid.minor = element_blank())+
            theme_classic()+
           theme(axis.title.y=element_blank(),
                 axis.title.x=element_blank(),
                 axis.text.y=element_text(size=8, color="black", family="sans"),
                 axis.text.x=element_text(angle=45,size=8,vjust=1, hjust=1, color="black", family="sans"),
                 axis.line.x = element_line(size = 0.4),
                 axis.ticks.x = element_blank(),
                 axis.ticks.y = element_line(size = 0.4),
                 axis.line.y = element_line(size = 0.4))+
           theme(legend.title = element_text(size=12),
                 legend.text = element_text(size=8),
                 legend.key.size = unit(0.5, 'cm'), #change legend key size
                 legend.key.height = unit(0.5, 'cm'), #change legend key height
                 legend.key.width = unit(0.5, 'cm'), #change legend key width
                 legend.position="right")+
           ggeasy::easy_center_title()

setwd(path_graphs)

svgname<-paste("Logpval_selected_order_from_REP",".svg",sep='')

ggsave(svgname,plot=logpval_dotplot, device ='svg', width=13, height=13)

# Tile plot

In [114]:
order_from_REP<-REP_genes$gene
        
        
cat("order_from_REP_0\n")
cat(str(order_from_REP))
cat("\n")



tile_plot<-DE_results_sel[which(DE_results_sel$gene%in%order_from_REP),]


if(DEBUG == 1){
  cat("tile_plot_0\n")
  cat(str(tile_plot))
  cat("\n")
}

tile_plot$gene<-factor(tile_plot$gene, levels=rev(order_from_REP), ordered=T)



tile_plot$SIG<-NA

tile_plot$SIG[which(tile_plot$minuslog10padj >= 1.3)]<-'YES'
tile_plot$SIG[which(tile_plot$minuslog10padj < 1.3)]<-'NO'


tile_plot$SIG<-factor(tile_plot$SIG, levels=c('NO','YES'), ordered=T)

if(DEBUG == 1){
  cat("tile_plot_1\n")
  cat(str(tile_plot))
  cat("\n")
}

order_from_REP_0
 chr [1:47] "CUX1" "RUNX1" "EZH2" "XRCC2" "ABI1" "ACTB" "ADAM10" "ANGPT1" ...

tile_plot_0
'data.frame':	368 obs. of  10 variables:
 $ gene          : chr  "PIP4K2A" "C3orf52" "MEF2C" "WASF2" ...
 $ baseMean      : num  301.1 6.6 686.9 137.4 33.6 ...
 $ log2FoldChange: num  0.21776 0.00297 0.0066 0.00122 -0.00421 ...
 $ lfcSE         : num  0.1375 0.0209 0.022 0.0203 0.0211 ...
 $ pvalue        : num  0.00255 0.01433 0.02977 0.75244 0.09663 ...
 $ padj          : num  0.758 0.867 0.986 0.996 0.996 ...
 $ contrast      : Ord.factor w/ 5 levels "Genotype_rs139141690_HET_vs_wt"<..: 1 1 1 1 1 1 1 1 1 1 ...
 $ identity      : Ord.factor w/ 13 levels "1"<"2"<"3"<"4"<..: 3 3 3 3 3 3 3 3 3 3 ...
 $ minuslog10padj: num  0.1203 0.0618 0.0063 0.0016 0.0016 ...
 $ abslogfc      : num  0.21776 0.00297 0.0066 0.00122 0.00421 ...

tile_plot_1
'data.frame':	368 obs. of  11 variables:
 $ gene          : Ord.factor w/ 47 levels "ZFPM1"<"WASF2"<..: 16 37 20 2 34 15 6 5 36 38 ...
 $ baseM

## Add the gene class information

In [115]:
tile_plot<-merge(tile_plot,
                 REP_genes,
                 by="gene")

if(DEBUG == 1){
  cat("tile_plot_2\n")
  cat(str(tile_plot))
  cat("\n")
}
                 

tile_plot_2
'data.frame':	368 obs. of  13 variables:
 $ gene          : Ord.factor w/ 47 levels "ZFPM1"<"WASF2"<..: 43 43 43 43 43 43 43 43 42 42 ...
 $ baseMean      : num  152 151 151 152 151 ...
 $ log2FoldChange: num  0.058556 0.272214 0.000448 0.249473 0.156426 ...
 $ lfcSE         : num  0.0533 0.0721 0.0202 0.0508 0.0786 ...
 $ pvalue        : num  1.26e-01 1.66e-05 9.11e-01 7.76e-08 8.88e-03 ...
 $ padj          : num  0.57136 0.00122 0.99632 0.00002 0.13649 ...
 $ contrast      : Ord.factor w/ 5 levels "Genotype_rs139141690_HET_vs_wt"<..: 2 4 1 4 2 1 3 3 4 2 ...
 $ identity      : Ord.factor w/ 13 levels "1"<"2"<"3"<"4"<..: 1 3 3 1 3 1 1 3 1 3 ...
 $ minuslog10padj: num  0.2431 2.9131 0.0016 4.6998 0.8649 ...
 $ abslogfc      : num  0.058556 0.272214 0.000448 0.249473 0.156426 ...
 $ SIG           : Ord.factor w/ 2 levels "NO"<"YES": 1 2 1 2 1 1 2 1 1 1 ...
 $ other_string  : chr  "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "GOBP_MEGAKARYOCYTE_DIF

## graph

In [116]:
 ggheatmap_DE <-ggplot(data=tile_plot,
                     aes(x=contrast, y=gene, fill = log2FoldChange))+
    geom_tile()+
    geom_tile(data=subset(tile_plot, SIG == 'NO'), fill = NA, color = "black", size = 0.1)+
    geom_tile(data=subset(tile_plot, SIG == 'YES'), fill = NA, color = "black", size = 1)+
    scale_fill_gradient2(name=paste("log2FC", sep="\n"),
                         low = "blue", high = "red",mid="white",midpoint=0,
                         na.value = NA)

ggheatmap_DE<-ggheatmap_DE+
                     theme_cowplot(font_size = 2,
                  font_family = "sans")+
                     facet_grid(Gene_class ~ identity ,
             scales='free_y', space='free_y', switch="y",
             labeller = labeller(Gene_class = function(labels) paste("Gene Class:", labels))) + # Add a label for the Gene_class facet
                    theme( strip.background = element_blank(),
                           strip.placement = "outside",
                           strip.text = element_text(size=5,color="black", family="sans"),
                           panel.spacing = unit(0.2, "lines"),
                           panel.background=element_rect(fill="white"),
                           panel.border=element_rect(colour="white",size=0,5),
                           panel.grid.major = element_blank(),
                           panel.grid.minor = element_blank())+
            theme_classic()+
           theme(axis.title.y=element_blank(),
                 axis.title.x=element_blank(),
                 axis.text.y=element_text(size=8, color="black", family="sans"),
                 axis.text.x=element_text(angle=45,size=8,vjust=1, hjust=1, color="black", family="sans"),
                 axis.line.x = element_line(size = 0.4),
                 axis.ticks.x = element_blank(),
                 axis.ticks.y = element_line(size = 0.4),
                 axis.line.y = element_line(size = 0.4))+
           theme(legend.title = element_text(size=12),
                 legend.text = element_text(size=8),
                 legend.key.size = unit(0.5, 'cm'), #change legend key size
                 legend.key.height = unit(0.5, 'cm'), #change legend key height
                 legend.key.width = unit(0.5, 'cm'), #change legend key width
                 legend.position="right")+
           ggeasy::easy_center_title()

In [117]:
setwd(path_graphs)

svgname<-paste("Tile_plot_selected_order_from_REP",".svg",sep='')

ggsave(svgname,plot=ggheatmap_DE, device ='svg')

Saving 7 x 7 in image


In [118]:
  path_graphs = paste("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/Figure_graphs/",sep='')
  
  if (file.exists(path_graphs)){
    
    
  }else{
    
    dir.create(file.path(path_graphs))
    
  }#path_Downstream_analysis

NULL

In [119]:
setwd(path_graphs)

svgname<-paste("Figure_5_panel_D_DE_part",".svg",sep='')

ggsave(svgname,plot=ggheatmap_DE, device ='svg', height=12, width=4)

# Lolliplot of ORA pathways

## Read in ORA result

In [120]:
ORA_result<-readRDS(file="/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/DE_per_cluster/ORA_global_results_significant_Diff_K562.rds")

ORA_result<-droplevels(ORA_result)

cat("ORA_result_0\n")
str(ORA_result)
cat("\n")

ORA_result_0
'data.frame':	1901 obs. of  15 variables:
 $ ID            : chr  "GSE15330_HSC_VS_MEGAKARYOCYTE_ERYTHROID_PROGENITOR_IKAROS_KO_DN" "GSE15330_LYMPHOID_MULTIPOTENT_VS_MEGAKARYOCYTE_ERYTHROID_PROGENITOR_UP" "GSE26290_CTRL_VS_AKT_INHIBITOR_TREATED_ANTI_CD3_AND_IL2_STIM_CD8_TCELL_DN" "RACCACAR_AML_Q6" ...
 $ Description   : chr  "GSE15330_HSC_VS_MEGAKARYOCYTE_ERYTHROID_PROGENITOR_IKAROS_KO_DN" "GSE15330_LYMPHOID_MULTIPOTENT_VS_MEGAKARYOCYTE_ERYTHROID_PROGENITOR_UP" "GSE26290_CTRL_VS_AKT_INHIBITOR_TREATED_ANTI_CD3_AND_IL2_STIM_CD8_TCELL_DN" "RACCACAR_AML_Q6" ...
 $ GeneRatio     : chr  "1/1" "1/1" "1/1" "1/1" ...
 $ BgRatio       : chr  "116/9747" "154/9747" "131/9747" "151/9747" ...
 $ RichFactor    : num  0.00862 0.00649 0.00763 0.00662 0.04348 ...
 $ FoldEnrichment: num  84 63.3 74.4 64.5 423.8 ...
 $ zScore        : num  9.11 7.89 8.57 7.97 20.56 ...
 $ pvalue        : num  0.0119 0.0158 0.01344 0.01549 0.00236 ...
 $ p.adjust      : num  0.0158 0.0158 0.0158 0.01549 0.0023

## Selected annotations

In [121]:
ORA_result_sel<-ORA_result[which(ORA_result$ID%in%c(annotations) &
                                ORA_result$identity %in%c('1','3') &
                                ORA_result$Count >=3),]

cat("ORA_result_sel_0\n")
str(ORA_result_sel)
cat("\n")

summary(as.factor(ORA_result_sel$ID))
cat("\n")

ORA_result_sel.dt<-data.table(ORA_result_sel, keys=c('ID','contrast','identity'))

ORA_result_sel_max<-as.data.frame(ORA_result_sel.dt[,.SD[which.max(minuslog10padj)], by=c('ID','contrast','identity')], stringsAsFactors=F)

cat("ORA_result_sel_max_0\n")
str(ORA_result_sel_max)
cat("\n")



ORA_result_sel_0
'data.frame':	18 obs. of  15 variables:
 $ ID            : chr  "AKT_UP_MTOR_DN.V1_UP" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING" "HP_ABNORMAL_PLATELET_VOLUME" ...
 $ Description   : chr  "AKT_UP_MTOR_DN.V1_UP" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING" "HP_ABNORMAL_PLATELET_VOLUME" ...
 $ GeneRatio     : chr  "3/4" "4/51" "11/146" "5/31" ...
 $ BgRatio       : chr  "99/9747" "41/9747" "176/9747" "21/9747" ...
 $ RichFactor    : num  0.0303 0.0976 0.0625 0.2381 0.25 ...
 $ FoldEnrichment: num  73.84 18.65 4.17 74.86 78.6 ...
 $ zScore        : num  14.76 8.21 5.24 19.14 17.55 ...
 $ pvalue        : num  4.04e-06 5.84e-05 6.45e-05 4.56e-09 1.48e-07 ...
 $ p.adjust      : num  8.07e-06 5.45e-04 2.00e-03 1.18e-08 2.76e-07 ...
 $ qvalue        : num  NA 2.33e-04 1.25e-03 9.59e-10 2.23e-08 ...
 $ Count         : int  3 4 11 5 4 4 5 4 4 11 ...
 $ minuslog10padj: num  5.09 3.26 2.7 7.93 6.56 ...
 $ geneID        : chr  "ADAM10/IL1R1/TOR3A" "P

AKT_UP_MTOR_DN.V1_UP GOBP_MEGAKARYOCYTE_DIFFERENTIATION 
                                 3                                  5 
       HP_ABNORMAL_PLATELET_VOLUME  HP_INCREASED_MEAN_PLATELET_VOLUME 
                                 4                                  2 
              WP_PI3KAKT_SIGNALING 
                                 4


ORA_result_sel_max_0
'data.frame':	15 obs. of  16 variables:
 $ ID            : chr  "AKT_UP_MTOR_DN.V1_UP" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING" "HP_ABNORMAL_PLATELET_VOLUME" ...
 $ contrast      : Ord.factor w/ 4 levels "Genotype_Del_80bp_vs_wt"<..: 2 2 2 1 1 1 3 3 1 1 ...
 $ identity      : Ord.factor w/ 13 levels "13"<"12"<"11"<..: 11 11 11 11 11 11 11 11 13 13 ...
 $ Description   : chr  "AKT_UP_MTOR_DN.V1_UP" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING" "HP_ABNORMAL_PLATELET_VOLUME" ...
 $ GeneRatio     : chr  "3/4" "4/51" "11/146" "5/31" ...
 $ BgRatio       : chr  "99/9747" "41/9747" "176/9747" "21/9747" ...
 $ RichFactor    : num  0.0303 0.0976 0.0625 0.2381 0.25 ...
 $ FoldEnrichment: num  73.84 18.65 4.17 74.86 78.6 ...
 $ zScore        : num  14.76 8.21 5.24 19.14 17.55 ...
 $ pvalue        : num  4.04e-06 5.84e-05 6.45e-05 4.56e-09 1.48e-07 ...
 $ p.adjust      : num  8.07e-06 5.45e-04 2.00e-03 1.18e-08 2.76e-07 ...
 $ qvalue        : 

## Reorder levels

In [122]:
new_order<-rev(levels(ORA_result_sel_max$contrast))

new_order


ORA_result_sel_max$contrast<-factor(ORA_result_sel_max$contrast,
                               levels=new_order,
                               ordered=T)

cat("ORA_result_sel_max_0\n")
str(ORA_result_sel_max)
cat("\n")

[1] "Genotype_rs139141690_HET_vs_wt" "Genotype_rs139141690_vs_wt"    
[3] "Genotype_Del_16bp_vs_wt"        "Genotype_Del_80bp_vs_wt"

ORA_result_sel_max_0
'data.frame':	15 obs. of  16 variables:
 $ ID            : chr  "AKT_UP_MTOR_DN.V1_UP" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING" "HP_ABNORMAL_PLATELET_VOLUME" ...
 $ contrast      : Ord.factor w/ 4 levels "Genotype_rs139141690_HET_vs_wt"<..: 3 3 3 4 4 4 2 2 4 4 ...
 $ identity      : Ord.factor w/ 13 levels "13"<"12"<"11"<..: 11 11 11 11 11 11 11 11 13 13 ...
 $ Description   : chr  "AKT_UP_MTOR_DN.V1_UP" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING" "HP_ABNORMAL_PLATELET_VOLUME" ...
 $ GeneRatio     : chr  "3/4" "4/51" "11/146" "5/31" ...
 $ BgRatio       : chr  "99/9747" "41/9747" "176/9747" "21/9747" ...
 $ RichFactor    : num  0.0303 0.0976 0.0625 0.2381 0.25 ...
 $ FoldEnrichment: num  73.84 18.65 4.17 74.86 78.6 ...
 $ zScore        : num  14.76 8.21 5.24 19.14 17.55 ...
 $ pvalue        : num  4.04e-06 5.84e-05 6.45e-05 4.56e-09 1.48e-07 ...
 $ p.adjust      : num  8.07e-06 5.45e-04 2.00e-03 1.18e-08 2.76e-07 ...
 $ qvalue    

In [123]:
#ORA_result_sel_max

## graph

In [124]:
levels_ID<-unique(as.character(ORA_result_sel_max$ID))

levels_ID

[1] "AKT_UP_MTOR_DN.V1_UP"               "GOBP_MEGAKARYOCYTE_DIFFERENTIATION"
[3] "WP_PI3KAKT_SIGNALING"               "HP_ABNORMAL_PLATELET_VOLUME"       
[5] "HP_INCREASED_MEAN_PLATELET_VOLUME"

In [125]:
levels_ID<-unique(as.character(ORA_result_sel_max$ID))
      
ORA_result_sel_max$DUMMY<-factor(ORA_result_sel_max$ID, 
                             levels=rev(levels_ID), ordered=T)

if(DEBUG ==1){

cat("ORA_result_sel_max_1\n")
str(ORA_result_sel_max)
cat("\n")
}

ORA_result_sel_max_1
'data.frame':	15 obs. of  17 variables:
 $ ID            : chr  "AKT_UP_MTOR_DN.V1_UP" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING" "HP_ABNORMAL_PLATELET_VOLUME" ...
 $ contrast      : Ord.factor w/ 4 levels "Genotype_rs139141690_HET_vs_wt"<..: 3 3 3 4 4 4 2 2 4 4 ...
 $ identity      : Ord.factor w/ 13 levels "13"<"12"<"11"<..: 11 11 11 11 11 11 11 11 13 13 ...
 $ Description   : chr  "AKT_UP_MTOR_DN.V1_UP" "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING" "HP_ABNORMAL_PLATELET_VOLUME" ...
 $ GeneRatio     : chr  "3/4" "4/51" "11/146" "5/31" ...
 $ BgRatio       : chr  "99/9747" "41/9747" "176/9747" "21/9747" ...
 $ RichFactor    : num  0.0303 0.0976 0.0625 0.2381 0.25 ...
 $ FoldEnrichment: num  73.84 18.65 4.17 74.86 78.6 ...
 $ zScore        : num  14.76 8.21 5.24 19.14 17.55 ...
 $ pvalue        : num  4.04e-06 5.84e-05 6.45e-05 4.56e-09 1.48e-07 ...
 $ p.adjust      : num  8.07e-06 5.45e-04 2.00e-03 1.18e-08 2.76e-07 ...
 $ qvalue    

In [126]:
breaks_gene_sets<-as.numeric(ORA_result_sel_max$DUMMY)
labels_gene_sets<-as.character(gsub("\\..+$","",ORA_result_sel_max$DUMMY))

labels_gene_sets

[1] "AKT_UP_MTOR_DN"                     "GOBP_MEGAKARYOCYTE_DIFFERENTIATION"
 [3] "WP_PI3KAKT_SIGNALING"               "HP_ABNORMAL_PLATELET_VOLUME"       
 [5] "HP_INCREASED_MEAN_PLATELET_VOLUME"  "GOBP_MEGAKARYOCYTE_DIFFERENTIATION"
 [7] "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING"              
 [9] "AKT_UP_MTOR_DN"                     "HP_ABNORMAL_PLATELET_VOLUME"       
[11] "GOBP_MEGAKARYOCYTE_DIFFERENTIATION" "WP_PI3KAKT_SIGNALING"              
[13] "AKT_UP_MTOR_DN"                     "GOBP_MEGAKARYOCYTE_DIFFERENTIATION"
[15] "WP_PI3KAKT_SIGNALING"

In [127]:
ORA_lolliplot<-ggplot(data=ORA_result_sel_max, 
                            aes(y=as.numeric(DUMMY),
                                x=minuslog10padj)) +
        geom_segment(data=ORA_result_sel_max,
                     aes(y=as.numeric(DUMMY),
                         yend=as.numeric(DUMMY),
                         x=0,
                         xend=minuslog10padj),
                     color='black',
                     size=0.8)+
        geom_point(size=5, stroke=1, shape=21, color='black', fill="white")+
        geom_text(data=ORA_result_sel_max,
                  aes(x=minuslog10padj, y=as.numeric(DUMMY), label=Count),color="black",size=2, family="sans",fontface="bold")
      
      
      ORA_lolliplot <-ORA_lolliplot+
        theme_cowplot(font_size = 2,
                      font_family = "sans")+
        facet_grid(. ~ contrast+identity, scales='free_x', space='free_x', switch="y", drop=TRUE)+
        theme( strip.background = element_blank(),
               strip.placement = "outside",
               strip.text = element_text(size=5,color="black", family="sans"),
               panel.spacing = unit(0.2, "lines"),
               panel.background=element_rect(fill="white"),
               panel.border=element_rect(colour="white",size=0,5),
               panel.grid.major = element_blank(),
               panel.grid.minor = element_blank())+
        scale_x_continuous(name='-log10pval')+
        scale_y_continuous(name=NULL, breaks=breaks_gene_sets,
                           labels=labels_gene_sets)+
        theme_classic()+
        theme(axis.title=element_blank(),
              axis.title.y=element_blank(),
              axis.title.x=element_text(size=8,color="black", family="sans"),
              axis.text.y=element_text(size=6,color="black", family="sans", face='bold'),
              axis.text.x=element_text(size=6,color="black", family="sans"),
              axis.line.x = element_line(size = 0.4),
              axis.ticks.x = element_line(size = 0.4),
              axis.ticks.y = element_line(size = 0.4),
              axis.line.y = element_line(size = 0.4))+
        theme(legend.title = element_blank(),
              legend.text = element_text(size=6),
              legend.key.size = unit(0.5, 'cm'), #change legend key size
              legend.key.height = unit(0.5, 'cm'), #change legend key height
              legend.key.width = unit(0.5, 'cm'), #change legend key width
              legend.position="bottom")+
        guides(fill=guide_legend(nrow=1,byrow=TRUE))

In [128]:
  path_graphs = paste("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/Figure_graphs/",sep='')
  
  if (file.exists(path_graphs)){
    
    
  }else{
    
    dir.create(file.path(path_graphs))
    
  }#path_Downstream_analysis

NULL

In [129]:
setwd(path_graphs)

svgname<-paste("Figure_5_panel_DE_ORA_part",".svg",sep='')

ggsave(svgname,plot=ORA_lolliplot, device ='svg', height=4, width=12)